In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
mhs = pd.read_csv("../data/processed/mhs_main_experiment_annotations_with_split.csv")

print("Loaded shape:", mhs.shape)
print("Unique comments:", mhs["comment_id"].nunique())
print(mhs["split"].value_counts())

Loaded shape: (49433, 53)
Unique comments: 19761
split
train         34413
test           7713
validation     7307
Name: count, dtype: int64


In [3]:

target_col = "hatespeech"

# Hatespeech has 3 classes: 0, 1, 2
label_values = [0, 1, 2]

In [4]:
def make_distribution(group, label_col, label_values):
    counts = group[label_col].value_counts(normalize=True)

    return pd.Series({
        f"{label_col}_{label}_prob": counts.get(label, 0.0)
        for label in label_values
    })

In [5]:
full_dist = (
    mhs.groupby("comment_id")
    .apply(lambda g: make_distribution(g, target_col, label_values))
    .reset_index()
)

full_dist.head()

,comment_id,hatespeech_0_prob,hatespeech_1_prob,hatespeech_2_prob
0,6,1.000000,0.0,0.000000
1,7,0.500000,0.0,0.500000
2,10,0.333333,0.0,0.666667
3,11,0.000000,0.5,0.500000
4,12,0.500000,0.0,0.500000


In [6]:
metadata = (
    mhs.groupby("comment_id")
    .agg(
        text_clean=("text_clean", "first"),
        split=("split", "first"),
        target_type=("target_type", "first"),
        is_women_targeted=("is_women_targeted", "max"),
        is_immigrant_targeted=("is_immigrant_targeted", "max"),
        annotation_count=("annotator_id", "count"),
        unique_annotators=("annotator_id", "nunique")
    )
    .reset_index()
)

metadata.head()

,comment_id,text_clean,split,target_type,is_women_targeted,is_immigrant_targeted,annotation_count,unique_annotators
0,6,First off you look cool as fuck! Anyway if we ...,train,women_only,1,0,2,2
1,7,\*points to posters asking for palestinian rig...,test,immigrant_only,0,1,2,2
2,10,"They'll come back in your plan, also. Plus we ...",train,immigrant_only,0,1,3,3
3,11,"eat my fuck, bitch",validation,women_only,1,0,2,2
4,12,She ugly anyways,train,women_only,1,0,2,2


In [7]:
survey_item_cols = [
    "sentiment",
    "respect",
    "insult",
    "humiliate",
    "status",
    "dehumanize",
    "violence",
    "genocide",
    "attack_defend",
    "hatespeech"
]

survey_means = (
    mhs.groupby("comment_id")[survey_item_cols]
    .mean()
    .add_prefix("mean_")
    .reset_index()
)

survey_means.head()

,comment_id,mean_sentiment,mean_respect,mean_insult,mean_humiliate,mean_status,mean_dehumanize,mean_violence,mean_genocide,mean_attack_defend,mean_hatespeech
0,6,3.500000,4.000000,3.500000,4.000000,3.000000,3.500000,2.500000,1.000000,3.500000,0.000000
1,7,3.500000,3.500000,1.500000,1.000000,2.500000,1.500000,2.000000,2.000000,3.500000,1.000000
2,10,3.333333,3.666667,3.333333,2.666667,3.333333,2.666667,3.666667,3.333333,3.666667,1.333333
3,11,4.000000,4.000000,3.000000,2.500000,2.000000,0.500000,0.500000,0.000000,3.000000,1.500000
4,12,3.500000,4.000000,3.500000,3.000000,2.500000,1.500000,0.500000,0.000000,3.000000,1.000000


In [8]:
def entropy_from_probs(row, prob_cols):
    probs = row[prob_cols].values.astype(float)
    probs = probs[probs > 0]

    if len(probs) == 0:
        return 0.0

    return -np.sum(probs * np.log2(probs))

prob_cols = [f"{target_col}_{label}_prob" for label in label_values]

full_dist["hatespeech_entropy"] = full_dist.apply(
    lambda row: entropy_from_probs(row, prob_cols),
    axis=1
)

full_dist.head()

,comment_id,hatespeech_0_prob,hatespeech_1_prob,hatespeech_2_prob,hatespeech_entropy
0,6,1.000000,0.0,0.000000,-0.000000
1,7,0.500000,0.0,0.500000,1.000000
2,10,0.333333,0.0,0.666667,0.918296
3,11,0.000000,0.5,0.500000,1.000000
4,12,0.500000,0.0,0.500000,1.000000


In [9]:
full_dist["hatespeech_majority_label"] = (
    full_dist[prob_cols]
    .idxmax(axis=1)
    .str.extract(r"hatespeech_(\d+)_prob")
    .astype(int)
)

full_dist.head()

,comment_id,hatespeech_0_prob,hatespeech_1_prob,hatespeech_2_prob,hatespeech_entropy,hatespeech_majority_label
0,6,1.000000,0.0,0.000000,-0.000000,0
1,7,0.500000,0.0,0.500000,1.000000,0
2,10,0.333333,0.0,0.666667,0.918296,2
3,11,0.000000,0.5,0.500000,1.000000,1
4,12,0.500000,0.0,0.500000,1.000000,0


In [10]:
comment_soft_labels = (
    metadata
    .merge(full_dist, on="comment_id", how="left")
    .merge(survey_means, on="comment_id", how="left")
)

print("Final comment-level shape:", comment_soft_labels.shape)
comment_soft_labels.head()

Final comment-level shape: (19761, 23)


,comment_id,text_clean,split,target_type,is_women_targeted,is_immigrant_targeted,annotation_count,unique_annotators,hatespeech_0_prob,hatespeech_1_prob,...,mean_sentiment,mean_respect,mean_insult,mean_humiliate,mean_status,mean_dehumanize,mean_violence,mean_genocide,mean_attack_defend,mean_hatespeech
0,6,First off you look cool as fuck! Anyway if we ...,train,women_only,1,0,2,2,1.000000,0.0,...,3.500000,4.000000,3.500000,4.000000,3.000000,3.500000,2.500000,1.000000,3.500000,0.000000
1,7,\*points to posters asking for palestinian rig...,test,immigrant_only,0,1,2,2,0.500000,0.0,...,3.500000,3.500000,1.500000,1.000000,2.500000,1.500000,2.000000,2.000000,3.500000,1.000000
2,10,"They'll come back in your plan, also. Plus we ...",train,immigrant_only,0,1,3,3,0.333333,0.0,...,3.333333,3.666667,3.333333,2.666667,3.333333,2.666667,3.666667,3.333333,3.666667,1.333333
3,11,"eat my fuck, bitch",validation,women_only,1,0,2,2,0.000000,0.5,...,4.000000,4.000000,3.000000,2.500000,2.000000,0.500000,0.500000,0.000000,3.000000,1.500000
4,12,She ugly anyways,train,women_only,1,0,2,2,0.500000,0.0,...,3.500000,4.000000,3.500000,3.000000,2.500000,1.500000,0.500000,0.000000,3.000000,1.000000


In [12]:
def subgroup_distribution(df, subgroup_col, subgroup_value, prefix):
    subgroup_df = df[df[subgroup_col] == subgroup_value].copy()

    dist = (
        subgroup_df.groupby("comment_id")
        .apply(lambda g: make_distribution(g, target_col, label_values))
        .reset_index()
    )

    rename_map = {
        f"{target_col}_{label}_prob": f"{prefix}_{target_col}_{label}_prob"
        for label in label_values
    }

    return dist.rename(columns=rename_map)

women_annotator_dist = subgroup_distribution(
    mhs[mhs["is_women_targeted"] == 1],
    subgroup_col="annotator_gender_group",
    subgroup_value="women",
    prefix="women_annotators"
)

men_annotator_dist = subgroup_distribution(
    mhs[mhs["is_women_targeted"] == 1],
    subgroup_col="annotator_gender_group",
    subgroup_value="men",
    prefix="men_annotators"
)

women_annotator_dist.head()

,comment_id,women_annotators_hatespeech_0_prob,women_annotators_hatespeech_1_prob,women_annotators_hatespeech_2_prob
0,6,1.0,0.0,0.0
1,11,0.0,1.0,0.0
2,12,0.5,0.0,0.5
3,26,1.0,0.0,0.0
4,29,1.0,0.0,0.0


In [13]:
ideology_groups = [
    "extremely_conservative",
    "conservative",
    "slightly_conservative",
    "neutral",
    "slightly_liberal",
    "liberal",
    "extremely_liberal",
    "no_opinion"
]

ideology_dists = []

for group in ideology_groups:
    dist = subgroup_distribution(
        mhs[mhs["is_immigrant_targeted"] == 1],
        subgroup_col="annotator_ideology_group",
        subgroup_value=group,
        prefix=f"ideology_{group}"
    )

    ideology_dists.append(dist)

print("Created ideology subgroup distributions:", len(ideology_dists))

Created ideology subgroup distributions: 8


In [14]:
subgroup_prob_cols = [
    col for col in comment_soft_labels.columns
    if col.endswith("_prob") and col not in prob_cols
]

comment_soft_labels[subgroup_prob_cols] = comment_soft_labels[subgroup_prob_cols].fillna(0.0)

comment_soft_labels.head()

,comment_id,text_clean,split,target_type,is_women_targeted,is_immigrant_targeted,annotation_count,unique_annotators,hatespeech_0_prob,hatespeech_1_prob,...,mean_sentiment,mean_respect,mean_insult,mean_humiliate,mean_status,mean_dehumanize,mean_violence,mean_genocide,mean_attack_defend,mean_hatespeech
0,6,First off you look cool as fuck! Anyway if we ...,train,women_only,1,0,2,2,1.000000,0.0,...,3.500000,4.000000,3.500000,4.000000,3.000000,3.500000,2.500000,1.000000,3.500000,0.000000
1,7,\*points to posters asking for palestinian rig...,test,immigrant_only,0,1,2,2,0.500000,0.0,...,3.500000,3.500000,1.500000,1.000000,2.500000,1.500000,2.000000,2.000000,3.500000,1.000000
2,10,"They'll come back in your plan, also. Plus we ...",train,immigrant_only,0,1,3,3,0.333333,0.0,...,3.333333,3.666667,3.333333,2.666667,3.333333,2.666667,3.666667,3.333333,3.666667,1.333333
3,11,"eat my fuck, bitch",validation,women_only,1,0,2,2,0.000000,0.5,...,4.000000,4.000000,3.000000,2.500000,2.000000,0.500000,0.500000,0.000000,3.000000,1.500000
4,12,She ugly anyways,train,women_only,1,0,2,2,0.500000,0.0,...,3.500000,4.000000,3.500000,3.000000,2.500000,1.500000,0.500000,0.000000,3.000000,1.000000


In [15]:
os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/mhs_comment_soft_labels.csv"

comment_soft_labels.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Final shape:", comment_soft_labels.shape)

Saved: ../data/processed/mhs_comment_soft_labels.csv
Final shape: (19761, 23)


In [16]:
train_soft = comment_soft_labels[comment_soft_labels["split"] == "train"].copy()
val_soft = comment_soft_labels[comment_soft_labels["split"] == "validation"].copy()
test_soft = comment_soft_labels[comment_soft_labels["split"] == "test"].copy()

train_soft.to_csv("../data/processed/train_soft_labels.csv", index=False)
val_soft.to_csv("../data/processed/validation_soft_labels.csv", index=False)
test_soft.to_csv("../data/processed/test_soft_labels.csv", index=False)

print("Train:", train_soft.shape)
print("Validation:", val_soft.shape)
print("Test:", test_soft.shape)

Train: (13832, 23)
Validation: (2964, 23)
Test: (2965, 23)


In [17]:
os.makedirs("../results/tables", exist_ok=True)

report_path = "../results/tables/soft_label_aggregation_summary.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("SOFT LABEL AGGREGATION SUMMARY\n")
    f.write("=" * 70 + "\n\n")

    f.write("Main target column: hatespeech\n")
    f.write("Distribution classes: 0, 1, 2\n\n")

    f.write("Dataset size:\n")
    f.write(f"- Annotation rows: {len(mhs)}\n")
    f.write(f"- Unique comments: {mhs['comment_id'].nunique()}\n")
    f.write(f"- Final comment-level rows: {len(comment_soft_labels)}\n\n")

    f.write("Split sizes:\n")
    f.write(str(comment_soft_labels["split"].value_counts()))
    f.write("\n\n")

    f.write("Mean hatespeech distribution columns:\n")
    f.write(str(comment_soft_labels[prob_cols].mean()))
    f.write("\n\n")

    f.write("Hatespeech entropy summary:\n")
    f.write(str(comment_soft_labels["hatespeech_entropy"].describe()))
    f.write("\n\n")

    f.write("Majority label distribution:\n")
    f.write(str(comment_soft_labels["hatespeech_majority_label"].value_counts().sort_index()))
    f.write("\n\n")

    f.write("Subgroup distribution columns created:\n")
    f.write(f"- Women annotator distribution columns: 3\n")
    f.write(f"- Men annotator distribution columns: 3\n")
    f.write(f"- Ideology groups: {len(ideology_groups)} groups x 3 columns each\n")

print("Saved report:", report_path)

Saved report: ../results/tables/soft_label_aggregation_summary.txt
